In [6]:
import requests
import xml.etree.ElementTree as ET
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
import time
import os
from dotenv import load_dotenv

load_dotenv(override = True)

SS_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY")  # ready for when the key arrives
GROQ_API_KEY = os.getenv("GROQ_API_KEY")            # we'll use this in Week 2
SD_API_KEY = os.getenv("SCIENCE_DIRECT_API_KEY")

TOPIC = "pre-strain toughness failure behavior high-strength steel damage mechanics"
MAX_PAPERS = 50

print("Setup ready!")
print("SS API Key loaded:", "✅" if SS_API_KEY else "⏳ waiting")
print("Groq API Key loaded:", "✅" if GROQ_API_KEY else "⏳ waiting")
print("ScienceDirect API Key loaded:", "✅" if SD_API_KEY else "⏳ waiting")



Setup ready!
SS API Key loaded: ✅
Groq API Key loaded: ✅
ScienceDirect API Key loaded: ✅


In [7]:
load_dotenv(override=True)

def fetch_sciencedirect(query, max_results=25):
    url = "https://api.elsevier.com/content/search/sciencedirect"
    headers = {
        "X-ELS-APIKey": SD_API_KEY,
        "Accept": "application/json"
    }
    params = {
        "query": query,
        "count": max_results,
        "field": "dc:title,dc:description,prism:coverDate,dc:creator,prism:url"
    }
    response = requests.get(url, headers=headers, params=params)
    print("ScienceDirect status:", response.status_code)
    
    papers = []
    data = response.json()
    entries = data.get("search-results", {}).get("entry", [])
    
    for entry in entries:
        abstract = entry.get("dc:description", "")
        title = entry.get("dc:title", "")
        if abstract and title:
            papers.append({
                "title": title,
                "abstract": abstract,
                "year": entry.get("prism:coverDate", "")[:4],
                "authors": [entry.get("dc:creator", "")],
                "url": entry.get("prism:url", ""),
                "source": "sciencedirect"
            })
    
    print(f"Fetched {len(papers)} papers from ScienceDirect")
    return papers

sd_papers = fetch_sciencedirect(TOPIC, max_results=25)
print("Sample title:", sd_papers[0]["title"] if sd_papers else "none found")

ScienceDirect status: 401
Fetched 0 papers from ScienceDirect
Sample title: none found


In [3]:
def fetch_semantic_scholar(query, max_results=25):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    headers = {
        "User-Agent": "LitResearchTool/1.0",
        "x-api-key": SS_API_KEY
    }
    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,abstract,year,authors,url"
    }
    response = requests.get(url, params=params, headers=headers)
    print("Semantic Scholar status:", response.status_code)
    data = response.json()
    print("Total results found:", data.get("total", 0))
    papers = []
    for p in data.get("data", []):
        if p.get("abstract"):
            papers.append({
                "title": p.get("title", ""),
                "abstract": p.get("abstract", ""),
                "year": p.get("year", ""),
                "authors": [a["name"] for a in p.get("authors", [])],
                "url": p.get("url", ""),
                "source": "semantic_scholar"
            })
    return papers

ss_papers = fetch_semantic_scholar("pre-strain toughness high-strength steel damage mechanics", max_results=25)
print(f"\nFetched {len(ss_papers)} papers from Semantic Scholar")
print("Sample title:", ss_papers[0]["title"] if ss_papers else "none found")

Semantic Scholar status: 200
Total results found: 873

Fetched 14 papers from Semantic Scholar
Sample title: Influence of pre-strain on fracture toughness of 3rd generation advanced high strength steels


In [4]:
# print("API Key loaded:", "✅" if SS_API_KEY else "❌ None — key not loading from .env!")
# print("Key preview:", SS_API_KEY[:8] + "..." if SS_API_KEY else "NONE")

In [5]:
# For now, all papers come from arXiv
# When Semantic Scholar key arrives, we'll merge both sources here
all_papers = arxiv_papers + ss_papers + sd_papers

# Deduplicate by title
seen_titles = set()
unique_papers = []
for p in all_papers:
    t = p["title"].lower().strip()
    if t not in seen_titles and len(t) > 10:
        seen_titles.add(t)
        unique_papers.append(p)

print(f"Unique papers: {len(unique_papers)}")

# Create one text chunk per paper (title + abstract)
def make_chunk(paper):
    return f"Title: {paper['title']}\nAbstract: {paper['abstract']}"

chunks = [make_chunk(p) for p in unique_papers]
print(f"\nSample chunk:\n{chunks[0][:300]}...")

NameError: name 'arxiv_papers' is not defined

In [ ]:
# Merge both sources
all_papers = arxiv_papers + ss_papers

# Deduplicate by title
seen_titles = set()
unique_papers = []
for p in all_papers:
    t = p["title"].lower().strip()
    if t not in seen_titles and len(t) > 10:
        seen_titles.add(t)
        unique_papers.append(p)

print(f"arXiv papers: {len(arxiv_papers)}")
print(f"Semantic Scholar papers: {len(ss_papers)}")
print(f"Total unique papers after deduplication: {len(unique_papers)}")

def make_chunk(paper):
    return f"Title: {paper['title']}\nYear: {paper.get('year', 'N/A')}\nAbstract: {paper['abstract']}"

chunks = [make_chunk(p) for p in unique_papers]
print(f"\nSample chunk:\n{chunks[0][:300]}...")

arXiv papers: 25
Semantic Scholar papers: 14
Total unique papers after deduplication: 39

Sample chunk:
Title: A damage-mechanics model for fracture nucleation and propagation
Year: 
Abstract: In this paper a composite model for earthquake rupture initiation and propagation is proposed. The model includes aspects of damage mechanics, fiber-bundle models, and slider-block models. An array of elements i...


In [ ]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings (may take ~30 seconds)...")
embeddings = model.encode(chunks, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"\n✅ FAISS index built: {index.ntotal} vectors, dimension {dim}")

# Save to disk
faiss.write_index(index, "papers.index")
with open("papers_metadata.pkl", "wb") as f:
    pickle.dump({"papers": unique_papers, "chunks": chunks}, f)

print("✅ Saved to papers.index and papers_metadata.pkl")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings (may take ~30 seconds)...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


✅ FAISS index built: 39 vectors, dimension 384
✅ Saved to papers.index and papers_metadata.pkl


In [ ]:
def retrieve(query, top_k=5):
    query_vec = model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i+1,
            "title": unique_papers[idx]["title"],
            "score": float(distances[0][i]),
            "snippet": unique_papers[idx]["abstract"][:200]
        })
    return results

# Test with a real research question
results = retrieve("How does pre-strain affect fracture toughness in high-strength steel?", top_k=5)

for r in results:
    print(f"\n#{r['rank']} {r['title']}")
    print(f"   {r['snippet']}...")


#1 Influence of pre-strain on fracture toughness of 3rd generation advanced high strength steels
   Abstract. The present work investigates the influence of pre-strain on the fracture toughness of 3rd Generation Advanced High Strength Steels (AHSS). Specifically, a Carbide Free Bainitic (CFB) and a ...

#2 Effects of pre-fatigue damage on mechanical properties of Q690 high-strength steel
   Abstract This paper investigates the residual strength of Q690 high-strength steel previously damaged by fatigue. Pre-fatigue damage was induced by subjecting the specimens to nine different loading c...

#3 Fracture Assessment of Weld Joints of High-Strength Steel in Pre-Strained Condition
   Unstable fractures tend to occur after ductile crack initiation or propagation. In most collapsed steel structures, a maximum 15% pre-strain was recorded, at the steel structural connections, during t...

#4 Revealing the Role of Pre-Strain on the Microstructure and Mechanical Properties of a High-Mn Austenit